# Stage C 03e — compact behavior controls and memory visualization

Run only after reviewing both 03d soaks. Set `RECURRENCE_POLICY` to `paper_exact` if its stability/resume gate passed; otherwise select `stabilized_rms_v1` and retain the intervention telemetry. This runs four distinct one-million-base checkpoints and traces the adaptive model on held-out validation streams.

In [ ]:
# USER CONFIGURATION — choose only after comparing 03d outputs
RECURRENCE_POLICY='paper_exact'  # or 'stabilized_rms_v1'
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='2cf1e9fc87025a71b10121ba9452c8175ec0fbb3'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
BUDGET_BASES=1_000_000; BLOCK_COUNT=4; D_MODEL=128; NUM_HEADS=4; HORIZON=3


In [ ]:
from pathlib import Path
from google.colab import drive
import json, subprocess, sys
mount=Path('/content/drive')
if not (mount/'MyDrive').is_dir(): drive.mount(str(mount),timeout_ms=120000)
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
selection=json.loads((Path(DRIVE_ROOT)/'runs/c1_tokenizers_cpu/tokenizer_selection.json').read_text())
dataset=Path(DRIVE_ROOT)/'stage_c_dataset/ordered_streams'/selection['selected_tokenizer']
PROTOCOL=repo/'studies/stage_c_ecoli_escherichia_paper_deep_memory_v2/protocol.json'
STUDY_ROOT=Path(DRIVE_ROOT)/'study/stage_c_ecoli_escherichia_paper_deep_memory_v2'
root=Path(DRIVE_ROOT)/f'runs/c15_paper_deep_compact_1m_{RECURRENCE_POLICY}'
subprocess.run(['seqtrainer-titans-stage-c-study','initialize','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)


In [ ]:
def run_logged(run_dir,label,command):
    try: subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',str(run_dir),'--label',label,'--repo',str(repo),'--',*command],check=True)
    except subprocess.CalledProcessError:
        for path in (run_dir/'FAILED.txt',run_dir/'logs'/f'{label}.log'):
            if path.exists(): print(path.read_text(errors='replace')[-16000:])
        raise
def deep_flags():
    flags=['--memory-architecture','paper_residual_mlp_v2','--memory-depth','2','--memory-expansion-factor','4','--memory-projection-convolution-kernel','4','--memory-normalize-queries-and-keys','--memory-gate-granularity','per_layer_channel','--memory-recurrence-policy',RECURRENCE_POLICY,'--memory-surprise-clip-norm','none','--memory-alpha-initial','0.001','--memory-eta-initial','0.9','--memory-theta-initial','0.001']
    return flags+(['--memory-associative-loss-reduction','sum','--memory-max-gradient-rms','none','--memory-max-gradient-rms-ratio','none','--memory-theta-max','1.0'] if RECURRENCE_POLICY=='paper_exact' else ['--memory-associative-loss-reduction','mean','--memory-max-gradient-rms','none','--memory-max-gradient-rms-ratio','10.0','--memory-theta-max','0.5'])
conditions={
 'deep_adaptive':('adaptive','compact_deep_adaptive_1m',deep_flags()),
 'deep_frozen':('frozen_memory','compact_deep_frozen_1m',deep_flags()),
 'no_memory':('no_memory','compact_no_memory_1m',deep_flags()),
 'shallow_adaptive':('adaptive','compact_shallow_adaptive_1m',['--memory-architecture','legacy_mlp_v1','--memory-depth','1','--memory-recurrence-policy','legacy_configured','--memory-surprise-clip-norm','none','--memory-associative-loss-reduction','mean','--memory-max-gradient-rms-ratio','10.0','--memory-theta-max','0.5','--memory-theta-initial','0.001']),
}
for name,(mode,run_id,memory_flags) in conditions.items():
    run_dir=root/name
    command=['seqtrainer-titans-stage-c-train','--dataset-dir',str(dataset),'--run-dir',str(run_dir),'--memory-mode',mode,'--horizon',str(HORIZON),'--batch-size','1','--max-valid-bases',str(BUDGET_BASES),'--checkpoint-every','250','--learning-rate','3e-5','--gradient-clip-norm','0.5','--validation-streams','8','--activation','float32','--block-count',str(BLOCK_COUNT),'--d-model',str(D_MODEL),'--num-heads',str(NUM_HEADS),'--persistent-tokens','4',*memory_flags,'--protocol',str(PROTOCOL),'--run-id',run_id]
    run_logged(run_dir,f'train_{name}',command)
    run_logged(run_dir,f'architecture_{name}',['seqtrainer-titans-stage-c-architecture','--checkpoint',str(run_dir/'latest.pt'),'--output-dir',str(run_dir)])
    print((run_dir/'MODEL_ARCHITECTURE.txt').read_text())
    subprocess.run(['seqtrainer-titans-stage-c-study','record','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--run-id',run_id,'--evidence-tier','confirmatory','--artifact',str(run_dir)],check=True)
print('Four independent compact conditions completed:',root)


In [ ]:
trace_dir=root/'deep_adaptive/memory_trace_val'
run_logged(trace_dir,'memory_trace',['seqtrainer-titans-stage-c-memory-trace','--dataset-dir',str(dataset),'--checkpoint',str(root/'deep_adaptive/latest.pt'),'--output-dir',str(trace_dir),'--split','val','--memory-mode','adaptive','--max-streams','8','--max-segments','128','--device','cuda','--protocol',str(PROTOCOL),'--run-id','deep_memory_trace_analysis'])
subprocess.run(['seqtrainer-titans-stage-c-study','record','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--run-id','deep_memory_trace_analysis','--evidence-tier','exploratory','--artifact',str(trace_dir)],check=True)
behavior=root/'deep_adaptive/controlled_memory_behavior.json'
run_logged(root/'deep_adaptive','controlled_memory_behavior',['seqtrainer-titans-stage-c-memory-behavior','--checkpoint',str(root/'deep_adaptive/latest.pt'),'--output',str(behavior),'--pairs','32','--device','cuda','--protocol',str(PROTOCOL),'--run-id','deep_controlled_behavior'])
subprocess.run(['seqtrainer-titans-stage-c-study','record','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--run-id','deep_controlled_behavior','--evidence-tier','exploratory','--artifact',str(behavior)],check=True)
validations={name:json.loads((root/name/'validation.json').read_text()) for name in conditions}
print(json.dumps({name:{'bpb':row['bits_per_base'],'update':row['memory_update_norm_mean'],'surprise':row['surprise_norm_mean']} for name,row in validations.items()},indent=2))
print('Visualization:',trace_dir/'memory_pca.svg')
print('Machine-readable trace:',trace_dir/'memory_trace.json')
print('Controlled nonlinear association probe:',behavior)
print('These compact results are a scale/no-scale gate, not a final memory-benefit claim.')
